# Refusal-direction steering on Tiny Aya (EN / HI / GU)

Lexsi AI Safety Hackathon, Track 2b. **SafeTune** extracts and applies the refusal direction, **AuditKIT** scores and compares the runs.

**Before you start**
1. `Runtime > Change runtime type > T4 GPU`.
2. Open https://huggingface.co/CohereLabs/tiny-aya-global and accept the licence (the model is gated).
3. Have a Hugging Face token with **write** access ready (https://huggingface.co/settings/tokens).

**How to run:** run the cells top to bottom, one at a time. One cell = one stage. Every stage writes its output to disk as it goes, and every stage can be re-run: finished work is skipped, an interrupted generation resumes.

If Colab disconnects: reconnect, run **Stage 0, 1, 2 and 4** again, then continue from the stage that was interrupted.

## Stage 0 · Config (edit this cell, then run it)

In [ ]:
# ---- EDIT THESE ---------------------------------------------------------------
REPO_URL   = "https://github.com/YOUR_USER/YOUR_REPO.git"   # the repo this notebook lives in
HF_USER    = "YOUR_HF_USERNAME"                             # Hub namespace the results are pushed to
# ---- defaults that should just work -------------------------------------------
MODEL      = "CohereLabs/tiny-aya-global"
ALPHAS     = "0.05,0.1,0.15,0.3,0.5,0.8"   # steering strengths; 0.05 and 0.1 added because 0.3 over-refused in the CPU dry run
LANGS      = "en,hi,gu"
PICK_LAYER = None                   # None = SafeTune's sweep picks the layer. Set 24 if Stage 6 says "fallback".
RUN_NAME   = "tiny-aya-steer-v1"    # change it to start a fresh run next to the old one
USE_DRIVE  = True                   # keep results on Google Drive so a disconnect loses nothing
PRIVATE    = True                   # Hub repos are created private; set False when you are ready to submit
# -------------------------------------------------------------------------------

assert "YOUR_USER" not in REPO_URL, "Edit REPO_URL at the top of this cell."
assert HF_USER != "YOUR_HF_USERNAME", "Edit HF_USER at the top of this cell."

import os
REPO_DIR = "/content/lexsi-steering"
RUN_ROOT = "/content/results"
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        RUN_ROOT = "/content/drive/MyDrive/lexsi_steering_runs"
    except Exception as e:
        print("Drive not mounted, results stay on the Colab disk:", e)
RUN_DIR = f"{RUN_ROOT}/{RUN_NAME}"
os.makedirs(RUN_DIR, exist_ok=True)
PICK_FLAG = f"--pick-layer {PICK_LAYER}" if PICK_LAYER is not None else ""
print("results go to:", RUN_DIR)

## Stage 1 · GPU check and clone the repo

In [ ]:
assert "RUN_DIR" in globals(), "Run Stage 0 (Config) first."
import os, subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True)
assert gpu.returncode == 0 and gpu.stdout.strip(), "No GPU. Runtime > Change runtime type > T4 GPU, then start again."
print("GPU:", gpu.stdout.strip())

if os.path.isdir(f"{REPO_DIR}/.git"):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
assert os.path.exists(f"{REPO_DIR}/src/eval_steering.py"), "Clone failed or wrong repo: src/eval_steering.py not found."
%cd {REPO_DIR}
!ls prompts/*/ | head -30

## Stage 2 · Install

Installs SafeTune and AuditKIT from PyPI and removes `torchao` (the SafeTune README asks for this on Colab).
If pip ends with a *"restart the runtime"* message: `Runtime > Restart session`, then run Stage 0, 1 and 2 again.

In [ ]:
assert "RUN_DIR" in globals(), "Run Stage 0 (Config) first."
%cd {REPO_DIR}
!pip install -q -U safetune auditkit datasets huggingface_hub 2>&1 | tail -5
!pip uninstall -y -q torchao
!pip list 2>/dev/null | grep -i -E "^(safetune|auditkit|torch|transformers|datasets|peft|torchao) " | tee {RUN_DIR}/pip_versions.txt
!python src/score.py --self-test | tail -1

## Stage 3 · Hugging Face login (token is typed into a hidden prompt)

In [ ]:
assert "RUN_DIR" in globals(), "Run Stage 0 (Config) first."
import os
from getpass import getpass
from huggingface_hub import login, hf_hub_download, whoami

if not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = getpass("Hugging Face token (write access): ").strip()
login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
print("logged in as:", whoami()["name"])
try:
    hf_hub_download(MODEL, "config.json")
    print("access to", MODEL, "OK")
except Exception as e:
    raise SystemExit(f"Cannot read {MODEL}. Accept the licence at https://huggingface.co/{MODEL} "
                     f"with the same account, then re-run this cell.\n{e}")

## Stage 4 · Smoke test (about 3 minutes)

Two prompts per set, English only, one alpha, on the real model. Catches install / access / memory problems before the long run.
After a disconnect, re-running this cell also re-enters the repo folder.

In [ ]:
assert os.environ.get("HF_TOKEN"), "Run Stage 3 (login) first."
%cd {REPO_DIR}
!python src/eval_steering.py --model {MODEL} --langs en --alpha 0.3 --limit 2 --n-val 4 --calib-n 16 \
        --max-new-tokens 48 --out-dir {RUN_DIR}/smoke
!python src/score.py --run-dir {RUN_DIR}/smoke | tail -25

## Stage 5 · Calibrate: extract the refusal direction

SafeTune's validation sweep scores every layer and picks one. Read the last lines of the output:

* `validation sweep ...` → good, continue.
* `fallback` → the sweep could not decide and took the middle layer. Set `PICK_LAYER = 24` in Stage 0, re-run Stage 0, then run **this** cell again (it recalibrates on its own when the layer setting changed).

In [ ]:
assert os.environ.get("HF_TOKEN"), "Run Stage 3 (login) first."
%cd {REPO_DIR}
!python src/eval_steering.py --model {MODEL} --langs {LANGS} --alpha {ALPHAS} {PICK_FLAG} \
        --stage calibrate --out-dir {RUN_DIR}
import json
cal = json.load(open(f"{RUN_DIR}/calibration.json"))
print(f"\nlayer {cal['layer']} of {cal['n_layers']}  |  reason: {cal['layer_reason']}  |  fell back: {cal['sweep_fell_back_to_middle']}")
print("residual norm at that layer:", cal["resid_norm_last_token_by_layer"][str(cal["layer"])])

## Stage 6 · Generate: baseline + one pass per alpha (the long one, roughly 30-50 minutes on a T4)

Answers are appended to `generations/<condition>.jsonl` one by one. If it stops, just run the cell again: it resumes.

In [ ]:
assert os.environ.get("HF_TOKEN"), "Run Stage 3 (login) first."
%cd {REPO_DIR}
!python src/eval_steering.py --model {MODEL} --langs {LANGS} --alpha {ALPHAS} {PICK_FLAG} \
        --with-ablation --out-dir {RUN_DIR}
!wc -l {RUN_DIR}/generations/*.jsonl

## Stage 7 · Score (SafeTune outputs → multilingual refusal classifier → AuditKIT compare)

In [ ]:
%cd {REPO_DIR}
!python src/score.py --run-dir {RUN_DIR} > {RUN_DIR}/score_stdout.txt 2>&1; tail -3 {RUN_DIR}/score_stdout.txt
from IPython.display import Markdown, display
display(Markdown(open(f"{RUN_DIR}/metrics.md", encoding="utf-8").read()))

## Stage 8 · Look at the answers before trusting the table

The classifier is a phrase match. Skim a few rows per language: refusals it missed, and "refusals" that are not.

In [ ]:
import json, random
def show(cond, lang, pset, n=4, seed=0):
    rows = [json.loads(l) for l in open(f"{RUN_DIR}/scored/{cond}.jsonl", encoding="utf-8")]
    rows = [r for r in rows if r["lang"] == lang and r["set"] == pset]
    random.Random(seed).shuffle(rows)
    for r in rows[:n]:
        print(f"[{cond} | {lang} | {pset}] refusal={r['refusal']} degenerate={r['degenerate']}")
        print("  Q:", r["prompt"][:160]); print("  A:", r["response"][:300].replace("\n", " ")); print()

best = (json.load(open(f"{RUN_DIR}/metrics.json")).get("best") or {}).get("condition", "alpha_0.30")
for lang in LANGS.split(","):
    show("baseline", lang, "harmful", n=3)
    show(best, lang, "harmful", n=3)
    show(best, lang, "benign_edgy", n=3)

## Stage 9 · Push everything to the Hugging Face Hub

Creates (or updates) four repos under your namespace:

| Repo | Type | Content |
|---|---|---|
| `tiny-aya-refusal-steering-results` | dataset | the whole run folder: generations, scored rows, metrics, AuditKIT runs, logs |
| `safety-prompts-en-hi-gu` | dataset | the prompt sets used here |
| `tiny-aya-refusal-direction` | model | `direction.pt` (the steering artifact) + how to load it |
| `tiny-aya-steering-auditkit-<condition>` | dataset | AuditKIT `RunResult` rows for baseline and the best alpha, pushed by AuditKIT itself |

Repos are private while `PRIVATE = True` (Stage 0). The results contain model answers to harmful prompts: read them before you make anything public.

In [ ]:
assert os.environ.get("HF_TOKEN"), "Run Stage 3 (login) first."
%cd {REPO_DIR}
import json
from huggingface_hub import HfApi
api, tok_ = HfApi(), os.environ["HF_TOKEN"]
metrics = json.load(open(f"{RUN_DIR}/metrics.json"))
best = (metrics.get("best") or {}).get("condition")
md_table = open(f"{RUN_DIR}/metrics.md", encoding="utf-8").read()
TAGS = "tags:\n- safety\n- refusal-steering\n- safetune\n- auditkit\n- tiny-aya\n- not-for-all-audiences\nlanguage:\n- en\n- hi\n- gu\n"

def push(repo, repo_type, folder, card, ignore=None):
    rid = f"{HF_USER}/{repo}"
    api.create_repo(rid, repo_type=repo_type, private=PRIVATE, exist_ok=True, token=tok_)
    api.upload_folder(repo_id=rid, repo_type=repo_type, folder_path=folder, token=tok_, ignore_patterns=ignore or [])
    api.upload_file(path_or_fileobj=card.encode(), path_in_repo="README.md", repo_id=rid, repo_type=repo_type, token=tok_)
    print("pushed", f"https://huggingface.co/{'datasets/' if repo_type == 'dataset' else ''}{rid}")

# 1. full run folder
push("tiny-aya-refusal-steering-results", "dataset", RUN_DIR,
     f"---\n{TAGS}---\n# Refusal-direction steering on {MODEL}\n\nSafeTune steering, scored with AuditKIT. "
     f"Run `{RUN_NAME}`.\n\n{md_table}", ignore=["smoke/*", "direction.pt"])

# 2. prompt sets
card = (f"---\n{TAGS}license: apache-2.0\n---\n# Safety prompt sets (EN / HI / GU)\n\n"
        "Per language: 40 harmful, 20 benign-but-edgy (XSTest-style), 15 capability prompts, plus calibration sets.\n"
        "EN and HI harmful prompts are sampled from CohereLabs/aya_redteaming (Apache-2.0); GU harmful prompts are "
        "translations of the EN set; everything else is author-curated (LLM-assisted drafts). See `src/build_prompts.py` in the code repo.\n")
push("safety-prompts-en-hi-gu", "dataset", "prompts", card)

# 3. the steering artifact
import shutil, tempfile
tmp = tempfile.mkdtemp(); shutil.copy(f"{RUN_DIR}/direction.pt", tmp); shutil.copy(f"{RUN_DIR}/calibration.json", tmp)
cal = json.load(open(f"{RUN_DIR}/calibration.json"))
card = (f"---\nbase_model: {MODEL}\n{TAGS}---\n# Refusal direction for {MODEL}\n\n"
        f"Unit vector from layer {cal['layer']} of {cal['n_layers']} ({cal['layer_reason']}), extracted with SafeTune "
        "`extract_refusal_direction`. No weights are changed.\n\n```python\nimport torch\n"
        "from safetune.steer import RefusalDirectionModel\nb = torch.load('direction.pt')\n"
        f"steered = RefusalDirectionModel(model, direction=b['direction'], mode='steer', strength={(best or 'alpha_0.30').split('_')[-1]})\n"
        "with steered:\n    out = model.generate(**inputs)\n```\n")
push("tiny-aya-refusal-direction", "model", tmp, card)

# 4. AuditKIT's own publisher for the baseline and the best steered run
import auditkit as ak
for cond in filter(None, ["baseline", best]):
    res = ak.RunResult.load(f"{RUN_DIR}/auditkit/run_{cond}.json")
    name = f"{HF_USER}/tiny-aya-steering-auditkit-{cond.replace('_', '-').replace('.', 'p')}"
    print("auditkit push:", res.push_to_hub(name, private=PRIVATE, token=tok_))

## Stage 10 · (optional, Day 2) Where does the direction live? Safety neurons

In [ ]:
%cd {REPO_DIR}
!python src/interpret_run.py --run-dir {RUN_DIR} --activation --per-lang

## Stage 11 · (optional) Download the run as a zip

In [ ]:
import shutil
from google.colab import files
zip_path = shutil.make_archive(f"/content/{RUN_NAME}", "zip", RUN_DIR)
files.download(zip_path)